In [ ]:
import requests
import numpy as np
from scipy.stats import poisson

TEAM_CACHE = {}

# -----------------------------
# SETTINGS
# -----------------------------
SEASONS = ["20252026"]
RECENT_GAMES = 12
MAX_GOALS = 15

# -----------------------------
# FETCH TEAM GAMES (ROBUST)
# -----------------------------
def get_team_games(team, seasons):
    if team in TEAM_CACHE:
        return TEAM_CACHE[team]

    games = []

    for season in seasons:
        url = f"https://api-web.nhle.com/v1/club-schedule-season/{team}/{season}"
        try:
            res = requests.get(url)
            if res.status_code != 200:
                continue
            data = res.json()
        except:
            continue

        for g in data.get("games", []):
            if g.get("gameState") != "OFF":
                continue

            games.append({
                "date": g["gameDate"],
                "home": g["homeTeam"]["abbrev"],
                "away": g["awayTeam"]["abbrev"],
                "home_goals": g["homeTeam"].get("score", 0),
                "away_goals": g["awayTeam"].get("score", 0),
                "result_type": g.get("gameOutcome", {}).get("lastPeriodType")
            })

    games.sort(key=lambda x: x["date"])
    TEAM_CACHE[team] = games
    return games

# -----------------------------
# RECENT FORM (SAFE, CAPPED & NORMALIZED)
# -----------------------------
def get_recent_stats(team, games, all_games_lookup, league_avg):
    recent = games[-RECENT_GAMES:]

    adj_gf = adj_ga = 0

    for g in recent:
        if g["home"] == team:
            opp = g["away"]
            gf = g["home_goals"]
            ga = g["away_goals"]
        else:
            opp = g["home"]
            gf = g["away_goals"]
            ga = g["home_goals"]

        opp_games = all_games_lookup[opp]
        opp_def = np.mean(
            [
                x["home_goals"] if x["away"] == opp else x["away_goals"]
                for x in opp_games[-RECENT_GAMES:]
            ]
        )
        opp_def = max(opp_def, 0.5)

        # Adjust goals relative to opponent defense and league avg
        game_adj_gf = gf * (league_avg / opp_def)
        game_adj_ga = ga * (league_avg / opp_def)

        # Cap extreme games
        game_adj_gf = min(game_adj_gf, 7)
        game_adj_ga = min(game_adj_ga, 7)

        adj_gf += game_adj_gf
        adj_ga += game_adj_ga

    n = len(recent)
    avg_gf = adj_gf / n
    avg_ga = adj_ga / n

    return avg_gf, avg_ga

# -----------------------------
# LAMBDA ESTIMATION (FIXED)
# -----------------------------
def estimate_lambdas(team1, team2, seasons):
    games1 = get_team_games(team1, seasons)
    games2 = get_team_games(team2, seasons)

    if len(games1) == 0 or len(games2) == 0:
        raise ValueError("No games found")

    # Build league dataset
    all_teams = set(
        [g['home'] for g in games1] +
        [g['away'] for g in games1] +
        [g['home'] for g in games2] +
        [g['away'] for g in games2]
    )

    all_games_lookup = {}
    for t in all_teams:
        all_games_lookup[t] = get_team_games(t, seasons)

    all_games = []
    for g in all_games_lookup.values():
        all_games.extend(g)

    # ✅ FIX 1: Correct league baseline
    league_avg_total = np.mean([g["home_goals"] + g["away_goals"] for g in all_games])
    league_avg = league_avg_total / 2  # per team

    # -----------------------------
    # TEAM STRENGTHS
    # -----------------------------
    # The 'team_strength' function was incomplete and not used. It has been removed.

    t1_gf, t1_ga = get_recent_stats(team1, games1, all_games_lookup, league_avg)
    t2_gf, t2_ga = get_recent_stats(team2, games2, all_games_lookup, league_avg)

    # -----------------------------
    # NORMALIZE (no double counting)
    # -----------------------------
    attack1 = t1_gf / league_avg
    defense1 = t1_ga / league_avg

    attack2 = t2_gf / league_avg
    defense2 = t2_ga / league_avg

    # -----------------------------
    # REGRESSION (stronger)
    # -----------------------------
    games_played = len(games1)
    reg = max(0.2, 0.6 - games_played * 0.01)

    attack1 = (1 - reg) * attack1 + reg * 1
    defense1 = (1 - reg) * defense1 + reg * 1

    attack2 = (1 - reg) * attack2 + reg * 1
    defense2 = (1 - reg) * defense2 + reg * 1

    # -----------------------------
    # MATCHUP LAMBDAS
    # -----------------------------
    lam1 = league_avg * attack1 * defense2
    lam2 = league_avg * attack2 * defense1

    # -----------------------------
    # HOME ICE (more realistic)
    # -----------------------------
    HOME_ADV = 1.035
    lam1 *= HOME_ADV

    # -----------------------------
    # TOTAL DAMPENER (VERY IMPORTANT)
    # -----------------------------
    total = lam1 + lam2
    target_total = league_avg_total

    shrink = 0.85  # controls overestimation
    adj_total = shrink * total + (1 - shrink) * target_total

    scale = adj_total / total
    lam1 *= scale
    lam2 *= scale
    lam1 = min(lam1, 4.5)
    lam2 = min(lam2, 4.5)

    return lam1, lam2, games1, games2


# -----------------------------
# FAST PROBABILITY ENGINE
# -----------------------------
def compute_matrix(lam1, lam2):
    VAR_MULT = 1.25  # 1.2–1.35 is a good range

    lam1_adj = lam1 * VAR_MULT
    lam2_adj = lam2 * VAR_MULT

    goals = np.arange(0, MAX_GOALS + 1)
    p1 = poisson.pmf(goals, lam1_adj)
    p2 = poisson.pmf(goals, lam2_adj)

    return goals, np.outer(p1, p2)

def match_probabilities(lam1, lam2):
    goals, matrix = compute_matrix(lam1, lam2)

    home = np.sum(np.tril(matrix, -1))
    draw = np.sum(np.diag(matrix))
    away = np.sum(np.triu(matrix, 1))

    total = home + draw + away
    if total == 0:
        return 0.33, 0.34, 0.33

    return home / total, draw / total, away / total

def puck_line_probs_by_team(lam1, lam2):
    goals, matrix = compute_matrix(lam1, lam2)
    diff = goals[:, None] - goals[None, :]

    # Each team's -1.5 probability
    team1_minus = np.sum(matrix[diff >= 2])
    team2_minus = np.sum(matrix[diff <= -2])

    # Each team's +1.5 probability
    team1_plus = 1 - team1_minus
    team2_plus = 1 - team2_minus

    return {
        "team1_-1.5": team1_minus,
        "team1_+1.5": team1_plus,
        "team2_-1.5": team2_minus,
        "team2_+1.5": team2_plus
    }

def total_probabilities(lam1, lam2, line):
    goals, matrix = compute_matrix(lam1, lam2)
    totals = goals[:, None] + goals[None, :]

    over = np.sum(matrix[totals > line])
    under = np.sum(matrix[totals < line])

    total = over + under
    if total == 0:
        return 0.5, 0.5

    over_prob = over / total
    under_prob = under / total

    # 🔥 FIX 8: Skew adjustment
    TOTAL_SKEW = 0.10
    over_prob = over_prob + TOTAL_SKEW * (over_prob - 0.5)
    over_prob = min(max(over_prob, 0.05), 0.95)
    under_prob = 1 - over_prob

    return over_prob, under_prob


# -----------------------------
# OT WIN RATE
# -----------------------------
def compute_ot_win_rate(team, games):
    ot_games = wins = 0

    for g in games:
        if g["result_type"] in ["OT", "SO"]:
            ot_games += 1

            if g["home"] == team:
                t1 = g["home_goals"]
                t2 = g["away_goals"]
            else:
                t1 = g["away_goals"]
                t2 = g["home_goals"]

            if t1 > t2:
                wins += 1

    if ot_games == 0:
        return 0.5

    return wins / ot_games


# -----------------------------
# ODDS + EV
# -----------------------------
def american_to_decimal(odds):
    return 1 + (odds / 100 if odds > 0 else 100 / abs(odds))

def compute_edge_and_ev(prob, odds):
    implied = 1 / odds
    edge = prob - implied
    ev = prob * odds - 1
    return edge, ev


# -----------------------------
# RUN
# -----------------------------
team1 = input("Home Team (e.g. BOS): ").strip().upper()
team2 = input("Away Team (e.g. NYR): ").strip().upper()

lam1, lam2, games1, games2 = estimate_lambdas(team1, team2, SEASONS)

home_reg, draw_reg, away_reg = match_probabilities(lam1, lam2)

ot1 = compute_ot_win_rate(team1, games1)
ot2 = compute_ot_win_rate(team2, games2)

total_ot = ot1 + ot2
ot_strengthhome = 0.5 + 0.1 * (lam1 - lam2)
ot_strengthhome = min(max(ot_strengthhome, 0.4), 0.6)
home_final = home_reg + draw_reg * ot_strengthhome
ot_strengthaway = lam2 / (lam1 + lam2)
away_final = away_reg + draw_reg * ot_strengthaway

# Normalize
total = home_final + away_final
home_final /= total
away_final /= total

# Smooth + blend market
def smooth(p, k=0.05):
    return p * (1 - k) + 0.5 * k

home_final = smooth(home_final)
away_final = smooth(away_final)

# Use the correct function and extract probabilities for display
all_pl_probs = puck_line_probs_by_team(lam1, lam2)
home_puck = all_pl_probs["team1_-1.5"]
away_puck = all_pl_probs["team2_+1.5"]
home_puck = min(max(home_puck, 0.05), 0.95)
away_puck = min(max(away_puck, 0.05), 0.95)

# -----------------------------
# ODDS INPUT
# -----------------------------
print("\nEnter American odds:")

home_odds = american_to_decimal(float(input(f"{team1} ML odds: ")))
away_odds = american_to_decimal(float(input(f"{team2} ML odds: ")))

print("\nEnter puck line odds with team and spread (example: BOS -1.5 or NYR +1.5)")

pl_team = input("Team for puck line: ").strip().upper()
pl_spread = float(input("Spread (-1.5 or +1.5): "))
pl_odds = american_to_decimal(float(input("Odds: ")))

pl_probs = puck_line_probs_by_team(lam1, lam2)

if pl_team == team1:
    key = f"team1_{pl_spread:+}".replace("+", "+").replace("-", "-")
elif pl_team == team2:
    key = f"team2_{pl_spread:+}".replace("+", "+").replace("-", "-")
else:
    raise ValueError("Invalid team for puck line")

prob = pl_probs[key]
edge_pl, ev_pl = compute_edge_and_ev(prob, pl_odds)

total_line = float(input("Total line (e.g. 5.5): "))

over_odds = american_to_decimal(float(input("Over odds: ")))
under_odds = american_to_decimal(float(input("Under odds: ")))

# -----------------------------
# TOTAL PROBS
# -----------------------------
over_prob, under_prob = total_probabilities(lam1, lam2, total_line)

# -----------------------------
# EDGE + EV
# -----------------------------
edge_h, ev_h = compute_edge_and_ev(home_final, home_odds)
edge_a, ev_a = compute_edge_and_ev(away_final, away_odds)

edge_over, ev_over = compute_edge_and_ev(over_prob, over_odds)
edge_under, ev_under = compute_edge_and_ev(under_prob, under_odds)

# -----------------------------
# OUTPUT
# -----------------------------
print("\n--- BETS ---")
if ev_h > 0.03: print(f"BET ML: {team1}")
if ev_a > 0.03: print(f"BET ML: {team2}")

if ev_pl > 0.03:
    print(f"BET PL: {pl_team} {pl_spread}")
if ev_over > 0.03: print(f"BET TOTAL: Over {total_line}")
if ev_under > 0.03: print(f"BET TOTAL: Under {total_line}")

print("\n--- EDGES ---")
print(f"{team1} ML Edge: {edge_h:.3f} | EV: {ev_h:.3f}")
print(f"{team2} ML Edge: {edge_a:.3f} | EV: {ev_a:.3f}")
print(f"Over {total_line}: {over_prob:.3f} | Edge: {edge_over:.3f} | EV: {ev_over:.3f}")
print(f"Under {total_line}: {under_prob:.3f} | Edge: {edge_under:.3f} | EV: {ev_under:.3f}")
print("\n--- PUCK LINE ---")
print(f"{pl_team} {pl_spread}: Prob={prob:.3f} | Edge={edge_pl:.3f} | EV={ev_pl:.3f}")


print("\n--- LAMBDAS ---")
print(f"{team1}: {lam1:.3f}")
print(f"{team2}: {lam2:.3f}")

print("\n--- FINAL WIN PROBABILITIES ---")
print(f"{team1}: {home_final:.3f}")
print(f"{team2}: {away_final:.3f}")

print("\n--- PUCK LINE ---")
print(f"{team1} -1.5: {home_puck:.3f}")
print(f"{team2} +1.5: {away_puck:.3f}")

print("\n--- TOTAL ---")
print(f"Expected Total: {lam1 + lam2:.3f}")

print("\n--- IMPLIED VS MODEL ---")
print(f"{team1} ML implied: {1/home_odds:.3f} vs model: {home_final:.3f}")
print(f"{team2} ML implied: {1/away_odds:.3f} vs model: {away_final:.3f}")

Home Team (e.g. BOS): SJS
Away Team (e.g. NYR): STL

Enter American odds:
SJS ML odds: -110
STL ML odds: -110

Enter puck line odds with team and spread (example: BOS -1.5 or NYR +1.5)
Team for puck line: SJS
Spread (-1.5 or +1.5): -1.5
Odds: 225
Total line (e.g. 5.5): 5.5
Over odds: -130
Under odds: 110

--- BETS ---
BET ML: STL
BET TOTAL: Over 5.5

--- EDGES ---
SJS ML Edge: -0.343 | EV: -0.654
STL ML Edge: 0.295 | EV: 0.564
Over 5.5: 0.745 | Edge: 0.179 | EV: 0.317
Under 5.5: 0.255 | Edge: -0.221 | EV: -0.463

--- PUCK LINE ---
SJS -1.5: Prob=0.062 | Edge=-0.246 | EV=-0.800

--- LAMBDAS ---
SJS: 1.871
STL: 3.878

--- FINAL WIN PROBABILITIES ---
SJS: 0.181
STL: 0.819

--- PUCK LINE ---
SJS -1.5: 0.062
STL +1.5: 0.358

--- TOTAL ---
Expected Total: 5.748

--- IMPLIED VS MODEL ---
SJS ML implied: 0.524 vs model: 0.181
STL ML implied: 0.524 vs model: 0.819


In [ ]:
print(home_puck, away_puck)
print(lam1, lam2)

0.3382081131165114 0.765503246497286
3.0595644321711157 2.73024482243457
